# ENARES 2024 CRS04 — Stage 03
## NB03 · Exportación R, comparación y cierre
Versión corregida: usa el diccionario real, detecta el diseño muestral y elimina listas placeholder.


In [1]:
!pip install -q google-cloud-bigquery pandas pandas-gbq pyarrow db-dtypes openpyxl XlsxWriter tabulate
from google.colab import auth, drive
from google.cloud import bigquery
from datetime import datetime, timezone
from pathlib import Path
import pandas as pd, hashlib, os
auth.authenticate_user(); drive.mount('/content/drive')
PROJECT_ID='enares-2024-crs04'; LOCATION='US'; EXPECTED_ROWS=18807
ROOT_DRIVE=Path('/content/drive/MyDrive/ENARES_2024_PROJECT')
LOG_DIR=ROOT_DRIVE/'05Resultados'/'logs'/'stage03'; SQL_DIR=ROOT_DRIVE/'02SQL'; OUTPUT_DIR=ROOT_DRIVE/'04Outputs'; DOCS_DIR=ROOT_DRIVE/'docs'; R_DIR=ROOT_DRIVE/'03Scripts_R'
for d in [LOG_DIR,SQL_DIR,OUTPUT_DIR,DOCS_DIR,R_DIR]: d.mkdir(parents=True,exist_ok=True)
RUN_UTC=datetime.now(timezone.utc).isoformat(); client=bigquery.Client(project=PROJECT_ID,location=LOCATION)
display(client.query('SELECT CURRENT_DATE() AS fecha_actual').result().to_dataframe())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 3.7 MB/s eta 0:00:00
Mounted at /content/drive


,fecha_actual
0,2026-08-12


## 1. Exportación para R

In [2]:
A=PROJECT_ID+'.enares2024_crs04_analytical.analytical_crs04_adolescents'; df=client.query('SELECT * FROM `'+A+'`').result().to_dataframe()
if len(df)!=EXPECTED_ROWS: raise RuntimeError('Export incompleto')
p=OUTPUT_DIR/'analytical_crs04_adolescents_for_r.csv'; df.to_csv(p,index=False)
def fsha(path):
 h=hashlib.sha256()
 with open(path,'rb') as f:
  for chunk in iter(lambda:f.read(1024*1024),b''): h.update(chunk)
 return h.hexdigest()
man=pd.DataFrame([{'file':p.name,'rows':len(df),'columns':len(df.columns),'sha256':fsha(p),'created_utc':RUN_UTC,'github_allowed':False}]); man.to_csv(LOG_DIR/'stage3_r_export_manifest.csv',index=False); display(man)

,file,rows,columns,sha256,created_utc,github_allowed
0,analytical_crs04_adolescents_for_r.csv,18807,1407,ed780499c2dc7e54c1232fb802c53dcbc56247629697cd...,2026-08-12T01:08:07.185260+00:00,False


## 2. Scripts R

In [3]:

# ============================================================
# 2. SCRIPTS R — SIN PLACEHOLDERS
# ============================================================

dictionary_path = OUTPUT_DIR / "diccionario_indicadores.csv"

if not dictionary_path.exists():
    raise FileNotFoundError(
        "Falta diccionario_indicadores.csv. "
        "Ejecuta primero NB04 — Diccionario y Linaje."
    )

dictionary = pd.read_csv(dictionary_path)

required_dictionary_columns = {
    "indicator_name",
    "variable_type",
    "status",
}

missing_dictionary_columns = (
    required_dictionary_columns - set(dictionary.columns)
)

if missing_dictionary_columns:
    raise RuntimeError(
        "El diccionario no contiene: "
        + ", ".join(sorted(missing_dictionary_columns))
    )

actual_columns = set(df.columns)

dictionary["exists_in_export"] = (
    dictionary["indicator_name"].isin(actual_columns)
)

indicator_registry = dictionary.loc[
    dictionary["status"].astype(str).str.lower().eq("implemented")
    & dictionary["variable_type"].astype(str).str.lower().str.contains(
        "binary",
        regex=False,
    )
    & dictionary["exists_in_export"],
    [
        "indicator_name",
        "label",
        "module",
        "variable_type",
        "numerator",
        "denominator",
        "missing_rule",
    ],
].drop_duplicates(
    subset=["indicator_name"]
).sort_values(
    ["module", "indicator_name"]
).reset_index(drop=True)

if indicator_registry.empty:
    raise RuntimeError(
        "No se encontraron indicadores binarios implementados "
        "y presentes en la tabla analítica."
    )

registry_path = OUTPUT_DIR / "stage3_r_indicator_registry.csv"

indicator_registry.to_csv(
    registry_path,
    index=False,
)

indicator_registry.to_csv(
    LOG_DIR / "stage3_r_indicator_registry.csv",
    index=False,
)

display(indicator_registry)
print("Indicadores enviados a R:", len(indicator_registry))


# ------------------------------------------------------------
# Validación binaria real antes de generar R
# ------------------------------------------------------------

domain_rows = []

for indicator in indicator_registry["indicator_name"]:
    observed = pd.Series(df[indicator]).dropna()
    unique_values = sorted(pd.unique(observed).tolist())

    invalid_values = [
        value
        for value in unique_values
        if value not in [0, 1, False, True]
    ]

    domain_rows.append({
        "indicator_name": indicator,
        "non_null_rows": int(len(observed)),
        "unique_values": str(unique_values),
        "binary_domain_valid": len(invalid_values) == 0,
        "invalid_values": str(invalid_values),
    })

domain_validation = pd.DataFrame(domain_rows)

domain_validation.to_csv(
    LOG_DIR / "stage3_r_indicator_domain_validation.csv",
    index=False,
)

display(domain_validation)

if not domain_validation["binary_domain_valid"].all():
    raise RuntimeError(
        "Hay indicadores con valores fuera de 0/1. "
        "Revisa stage3_r_indicator_domain_validation.csv."
    )


# ------------------------------------------------------------
# Resolver variables reales del diseño muestral
# ------------------------------------------------------------

column_lookup = {
    str(column).lower(): str(column)
    for column in df.columns
}

design_candidates = {
    "weight": [
        "FACTOR_ALUMNOS",
        "FACTOR_ALUMNO",
        "FACTOR",
        "PESO",
        "PONDERADOR",
    ],
    "strata": [
        "estrato",
        "CCDD",
        "ESTRATO",
    ],
    "psu": [
        "conglomerado",
        "ID",
        "CONGLOMERADO",
        "UPM",
        "upm",
    ],
    "stage2": [
        "seleccion_aula",
        "SELECCION_AULA",
        "aula",
        "AULA",
    ],
    "stage3": [
        "seleccion_alumno",
        "SELECCION_ALUMNO",
        "alumno",
        "ALUMNO",
    ],
}

def resolve_column(candidates):
    for candidate in candidates:
        if candidate in df.columns:
            return candidate

        lowered = candidate.lower()

        if lowered in column_lookup:
            return column_lookup[lowered]

    return None

resolved_design = {
    role: resolve_column(candidates)
    for role, candidates in design_candidates.items()
}

design_resolution = pd.DataFrame([
    {
        "design_role": role,
        "resolved_column": column,
        "found": column is not None,
    }
    for role, column in resolved_design.items()
])

design_resolution.to_csv(
    LOG_DIR / "stage3_r_design_variable_resolution.csv",
    index=False,
)

display(design_resolution)

missing_roles = [
    role
    for role in ["weight", "strata", "psu"]
    if resolved_design[role] is None
]

if missing_roles:
    raise RuntimeError(
        "No se pudieron resolver las variables obligatorias "
        "del diseño muestral: "
        + ", ".join(missing_roles)
    )


# ------------------------------------------------------------
# Generar survey_design.R
# ------------------------------------------------------------

ids_columns = [resolved_design["psu"]]

if resolved_design["stage2"]:
    ids_columns.append(resolved_design["stage2"])

if resolved_design["stage3"]:
    ids_columns.append(resolved_design["stage3"])

ids_formula = "~" + "+".join(ids_columns)
strata_formula = "~" + resolved_design["strata"]
weight_formula = "~" + resolved_design["weight"]

data_path_r = (
    "/content/drive/MyDrive/ENARES_2024_PROJECT/"
    "04Outputs/analytical_crs04_adolescents_for_r.csv"
)

registry_path_r = (
    "/content/drive/MyDrive/ENARES_2024_PROJECT/"
    "04Outputs/stage3_r_indicator_registry.csv"
)

design_path_r = (
    "/content/drive/MyDrive/ENARES_2024_PROJECT/"
    "04Outputs/design_crs04.rds"
)

output_path_r = (
    "/content/drive/MyDrive/ENARES_2024_PROJECT/"
    "04Outputs/tabulados_crs04_long.csv"
)

survey_design_r = f"""
library(survey)
library(readr)

options(survey.lonely.psu = "adjust")

d <- read_csv(
  "{data_path_r}",
  show_col_types = FALSE
)

required <- c(
  "{resolved_design['weight']}",
  "{resolved_design['strata']}",
  "{resolved_design['psu']}"
)

missing_required <- setdiff(
  required,
  names(d)
)

if (length(missing_required) > 0) {{
  stop(
    paste(
      "Missing survey-design variables:",
      paste(missing_required, collapse = ", ")
    )
  )
}}

design_crs04 <- svydesign(
  ids = {ids_formula},
  strata = {strata_formula},
  weights = {weight_formula},
  data = d,
  nest = TRUE
)

saveRDS(
  design_crs04,
  "{design_path_r}"
)
""".strip()

tabulados_r = f"""
library(survey)
library(readr)
library(dplyr)
library(tibble)

design_crs04 <- readRDS(
  "{design_path_r}"
)

registry <- read_csv(
  "{registry_path_r}",
  show_col_types = FALSE
)

vars <- unique(
  registry$indicator_name
)

missing_vars <- setdiff(
  vars,
  names(design_crs04$variables)
)

if (length(missing_vars) > 0) {{
  stop(
    paste(
      "Indicators absent from design data:",
      paste(missing_vars, collapse = ", ")
    )
  )
}}

validate_binary <- function(v) {{
  observed <- unique(
    na.omit(
      design_crs04$variables[[v]]
    )
  )

  invalid <- setdiff(
    observed,
    c(0, 1, FALSE, TRUE)
  )

  if (length(invalid) > 0) {{
    stop(
      paste(
        "Non-binary indicator:",
        v,
        paste(invalid, collapse = ", ")
      )
    )
  }}
}}

invisible(
  lapply(
    vars,
    validate_binary
  )
)

tabulate_indicator <- function(v) {{
  formula_v <- as.formula(
    paste0("~", v)
  )

  estimate <- svymean(
    formula_v,
    design_crs04,
    na.rm = TRUE
  )

  ci <- confint(
    estimate
  )

  valid_rows <- !is.na(
    design_crs04$variables[[v]]
  )

  pct <- as.numeric(
    coef(estimate)
  ) * 100

  se_pp <- as.numeric(
    SE(estimate)
  ) * 100

  ci_low <- as.numeric(
    ci[, 1]
  ) * 100

  ci_high <- as.numeric(
    ci[, 2]
  ) * 100

  cv <- ifelse(
    pct == 0,
    NA_real_,
    se_pp / pct
  )

  tibble(
    indicator_id = v,
    dimension = "Nacional",
    categoria = "Total",
    pct = pct,
    es = se_pp,
    ci_low = ci_low,
    ci_high = ci_high,
    cv = cv,
    n_unw = sum(valid_rows)
  )
}}

out <- bind_rows(
  lapply(
    vars,
    tabulate_indicator
  )
)

write_csv(
  out,
  "{output_path_r}",
  na = ""
)
""".strip()

(R_DIR / "survey_design.R").write_text(
    survey_design_r,
    encoding="utf-8",
)

(R_DIR / "tabulados.R").write_text(
    tabulados_r,
    encoding="utf-8",
)

print(survey_design_r)
print()
print(tabulados_r)


,indicator_name,label,module,variable_type,numerator,denominator,missing_rule
0,cree_al_menos_un_mito,Believes at least one sexual-violence myth,3.1,binary composite indicator,At least one component = 1,At least one component observed,All NULL -> NULL
1,justifica_al_menos_una,Justifies at least one punishment form,3.1,binary composite indicator,At least one component = 1,At least one component observed,Both NULL -> NULL
2,justifica_castigo_docente,Justifies physical punishment by a teacher,3.1,binary indicator,C3P301_4 = 1,"C3P301_4 IN (1,2)",3 or NULL -> NULL
3,justifica_castigo_parental,Justifies physical punishment by parents/careg...,3.1,binary indicator,C3P301_5 = 1,"C3P301_5 IN (1,2)",3 or NULL -> NULL
4,mito_fuera_casa,Believes myth that sexual violence occurs outs...,3.1,binary indicator,C3P303_4 = 1,"C3P303_4 IN (1,2)",3 or NULL -> NULL
...,...,...,...,...,...,...,...
66,conoce_demuna,Knows DEMUNA,3.6,binary indicator,Yes response,Valid awareness responses,Invalid/not applicable -> NULL
67,recibio_ayuda_institucional_vs,Received institutional help,3.6,binary indicator,At least one institutional-help item = 1,Applicable institutional universe,Outside universe -> NULL
68,recibio_ayuda_vs,Received help after sexual violence,3.6,binary indicator,At least one qualifying help type = 1,Applicable help-seeking universe,Outside universe -> NULL
69,recibio_ayuda_vs_victimas,Received help among victims,3.6,binary conditional indicator,recibio_ayuda_vs = 1,SPSS victim universe,Outside victim universe -> NULL


Indicadores enviados a R: 71


,indicator_name,non_null_rows,unique_values,binary_domain_valid,invalid_values
0,cree_al_menos_un_mito,18771,"[0, 1]",True,[]
1,justifica_al_menos_una,18773,"[0, 1]",True,[]
2,justifica_castigo_docente,18627,"[0, 1]",True,[]
3,justifica_castigo_parental,18353,"[0, 1]",True,[]
4,mito_fuera_casa,18303,"[0, 1]",True,[]
...,...,...,...,...,...
66,conoce_demuna,18807,"[0, 1]",True,[]
67,recibio_ayuda_institucional_vs,296,"[0, 1]",True,[]
68,recibio_ayuda_vs,2525,"[0, 1]",True,[]
69,recibio_ayuda_vs_victimas,3424,"[0, 1]",True,[]


,design_role,resolved_column,found
0,weight,FACTOR_ALUMNOS,True
1,strata,CCDD,True
2,psu,ID,True
3,stage2,None,False
4,stage3,None,False


library(survey)
library(readr)

options(survey.lonely.psu = "adjust")

d <- read_csv(
  "/content/drive/MyDrive/ENARES_2024_PROJECT/04Outputs/analytical_crs04_adolescents_for_r.csv",
  show_col_types = FALSE
)

required <- c(
  "FACTOR_ALUMNOS",
  "CCDD",
  "ID"
)

missing_required <- setdiff(
  required,
  names(d)
)

if (length(missing_required) > 0) {
  stop(
    paste(
      "Missing survey-design variables:",
      paste(missing_required, collapse = ", ")
    )
  )
}

design_crs04 <- svydesign(
  ids = ~ID,
  strata = ~CCDD,
  weights = ~FACTOR_ALUMNOS,
  data = d,
  nest = TRUE
)

saveRDS(
  design_crs04,
  "/content/drive/MyDrive/ENARES_2024_PROJECT/04Outputs/design_crs04.rds"
)

library(survey)
library(readr)
library(dplyr)
library(tibble)

design_crs04 <- readRDS(
  "/content/drive/MyDrive/ENARES_2024_PROJECT/04Outputs/design_crs04.rds"
)

registry <- read_csv(
  "/content/drive/MyDrive/ENARES_2024_PROJECT/04Outputs/stage3_r_indicator_registry.csv",
  show_col_types = FALSE
)


## 3. Comparación SPSS vs R

In [4]:

# ============================================================
# 3. COMPARACIÓN SPSS VS R — CON TOLERANCIA
# ============================================================

TOLERANCE_PP = 0.01

sp = OUTPUT_DIR / "spss_reference_results.csv"
rp = OUTPUT_DIR / "tabulados_crs04_long.csv"

comparison_path = (
    LOG_DIR /
    "stage3_spss_vs_r_comparison.csv"
)

if sp.exists() and rp.exists():
    s = pd.read_csv(sp)
    r = pd.read_csv(rp)

    keys = [
        "indicator_id",
        "dimension",
        "categoria",
    ]

    required = set(keys + ["pct"])

    if not required.issubset(s.columns):
        raise RuntimeError(
            "spss_reference_results.csv no tiene "
            "las columnas requeridas."
        )

    if not required.issubset(r.columns):
        raise RuntimeError(
            "tabulados_crs04_long.csv no tiene "
            "las columnas requeridas."
        )

    comparison = s.merge(
        r[keys + ["pct"]],
        on=keys,
        how="outer",
        suffixes=("_spss", "_r"),
        indicator=True,
    )

    comparison["difference_pp"] = (
        comparison["pct_spss"]
        - comparison["pct_r"]
    )

    comparison["absolute_difference_pp"] = (
        comparison["difference_pp"].abs()
    )

    comparison["within_tolerance"] = (
        comparison["_merge"].eq("both")
        & comparison[
            "absolute_difference_pp"
        ].le(TOLERANCE_PP)
    )

    comparison.to_csv(
        comparison_path,
        index=False,
    )

    display(comparison)

    if not comparison[
        "within_tolerance"
    ].fillna(False).all():
        raise RuntimeError(
            "SPSS y R no coinciden dentro de "
            f"{TOLERANCE_PP} puntos porcentuales."
        )

else:
    print(
        "Pendiente: ejecutar survey_design.R y tabulados.R, "
        "y colocar spss_reference_results.csv en 04Outputs."
    )


Pendiente: ejecutar survey_design.R y tabulados.R, y colocar spss_reference_results.csv en 04Outputs.


## 4. Documentación y pass

In [5]:

# ============================================================
# 4. DOCUMENTACIÓN, EVIDENCIA Y PASS
# ============================================================

(DOCS_DIR / "stage3_description.md").write_text(
    "# Stage 03 CRS04\n\n"
    "BigQuery contiene las tablas cleaned y analytical.\n"
    "R reproduce los tabulados definidos por el diccionario "
    "real de indicadores.\n"
    "La comparación SPSS versus R usa tolerancia de "
    "0.01 puntos porcentuales.\n\n"
    f"Actualizado: {RUN_UTC}\n",
    encoding="utf-8",
)

evidence_patterns = {
    "stage2_prerequisite": [
        "*stage2*prerequisite*.csv",
    ],
    "key_validation": [
        "*key*validation*.csv",
    ],
    "cleaned_validation": [
        "*cleaned*validation*.csv",
    ],
    "indicator_domain_validation": [
        "*indicator*domain*validation*.csv",
        "stage3_r_indicator_domain_validation.csv",
    ],
    "lineage": [
        "*lineage*.csv",
    ],
    "r_export_manifest": [
        "stage3_r_export_manifest.csv",
    ],
    "spss_vs_r_comparison": [
        "stage3_spss_vs_r_comparison.csv",
    ],
}

evidence_rows = []

for evidence_name, patterns in evidence_patterns.items():
    matches = []

    for pattern in patterns:
        matches.extend(
            sorted(LOG_DIR.glob(pattern))
        )

    matches = list(
        dict.fromkeys(matches)
    )

    evidence_rows.append({
        "evidence_name": evidence_name,
        "found": len(matches) > 0,
        "matched_files": ";".join(
            str(path)
            for path in matches
        ),
    })

evidence = pd.DataFrame(evidence_rows)

evidence.to_csv(
    LOG_DIR / "stage3_nb03_evidence_inventory.csv",
    index=False,
)

display(evidence)

comparison_passed = False

if (
    LOG_DIR /
    "stage3_spss_vs_r_comparison.csv"
).exists():
    comparison_check = pd.read_csv(
        LOG_DIR /
        "stage3_spss_vs_r_comparison.csv"
    )

    comparison_passed = (
        len(comparison_check) > 0
        and comparison_check[
            "within_tolerance"
        ].fillna(False).all()
    )

required_evidence = evidence.loc[
    evidence["evidence_name"].isin([
        "r_export_manifest",
        "indicator_domain_validation",
        "lineage",
        "spss_vs_r_comparison",
    ])
]

passed = (
    len(df) == EXPECTED_ROWS
    and required_evidence["found"].all()
    and comparison_passed
)

(ROOT_DRIVE / "stage3_pass.md").write_text(
    "# Stage 03 Pass — CRS04\n\n"
    f"Resultado: {'PASS' if passed else 'NOT PASSED'}\n\n"
    f"Fecha UTC: {RUN_UTC}\n\n"
    "Tolerancia SPSS-R: 0.01 puntos porcentuales\n",
    encoding="utf-8",
)

closure = pd.DataFrame([{
    "run_utc": RUN_UTC,
    "analytical_rows": len(df),
    "expected_rows": EXPECTED_ROWS,
    "indicator_count": len(indicator_registry),
    "domain_validation_passed": bool(
        domain_validation[
            "binary_domain_valid"
        ].all()
    ),
    "comparison_passed": comparison_passed,
    "evidence_complete": bool(
        required_evidence["found"].all()
    ),
    "stage3_passed": passed,
}])

closure.to_csv(
    LOG_DIR / "stage3_nb03_closure.csv",
    index=False,
)

display(closure)
print("PASS" if passed else "NOT PASSED")


,evidence_name,found,matched_files
0,stage2_prerequisite,True,/content/drive/MyDrive/ENARES_2024_PROJECT/05R...
1,key_validation,True,/content/drive/MyDrive/ENARES_2024_PROJECT/05R...
2,cleaned_validation,True,/content/drive/MyDrive/ENARES_2024_PROJECT/05R...
3,indicator_domain_validation,True,/content/drive/MyDrive/ENARES_2024_PROJECT/05R...
4,lineage,True,/content/drive/MyDrive/ENARES_2024_PROJECT/05R...
5,r_export_manifest,True,/content/drive/MyDrive/ENARES_2024_PROJECT/05R...
6,spss_vs_r_comparison,False,


,run_utc,analytical_rows,expected_rows,indicator_count,domain_validation_passed,comparison_passed,evidence_complete,stage3_passed
0,2026-08-12T01:08:07.185260+00:00,18807,18807,71,True,False,False,False


NOT PASSED


## 5. Retrospectivas

In [6]:
from textwrap import dedent

retrospectives = {
    1: dedent(f"""
    # Sprint 1 Retrospective

    Fecha: {RUN_UTC}

    ## Qué se construyó

    - Traducción de la sintaxis SPSS de los módulos 3.1–3.4 a SQL para BigQuery.
    - Construcción de la tabla analytical_crs04_adolescents.
    - Validación estructural de filas, columnas y tipos de datos.
    - Implementación de controles automáticos de calidad para los indicadores.

    ## Qué aprendí

    - La traducción de SPSS a SQL requiere preservar exactamente las reglas de recodificación y los valores perdidos.
    - Es importante validar cada indicador inmediatamente después de implementarlo para evitar errores acumulados.

    ## Error técnico que no quiero repetir

    - Asumir equivalencias directas entre funciones de SPSS y SQL sin verificar la lógica de los valores SYSMIS y las condiciones IF.

    ## Riesgo mitigado

    - Se verificó que la tabla analítica conserva las {EXPECTED_ROWS:,} observaciones esperadas y que no existen duplicados después de la integración.
    """).strip(),

    2: dedent(f"""
    # Sprint 2 Retrospective

    Fecha: {RUN_UTC}

    ## Qué se construyó

    - Traducción de los módulos de acumulación de violencia, consecuencias y búsqueda de ayuda.
    - Implementación de indicadores compuestos y de polivictimización.
    - Validaciones automáticas del dominio de los indicadores (0/1/NULL).
    - Construcción del diccionario técnico de indicadores.

    ## Qué aprendí

    - Los indicadores compuestos requieren validar primero cada componente antes de agregarlos.
    - Un diccionario técnico facilita la trazabilidad entre SPSS, SQL y BigQuery.

    ## Error técnico que no quiero repetir

    - Mantener listas manuales de indicadores; ahora el registro se genera automáticamente desde el diccionario oficial.

    ## Riesgo mitigado

    - Todos los indicadores documentados fueron contrastados con el esquema real de la tabla analítica antes de generar los scripts de validación.
    """).strip(),

    3: dedent(f"""
    # Sprint 3 Retrospective

    Fecha: {RUN_UTC}

    ## Qué se construyó

    - Exportación reproducible hacia R.
    - Generación automática de survey_design.R y tabulados.R.
    - Comparación automatizada entre resultados de SPSS y R.
    - Inventario de evidencia, linaje, hashes SHA-256 y documentación técnica para el cierre de Stage 03.

    ## Qué aprendí

    - La reproducibilidad mejora cuando toda la documentación y los scripts se generan automáticamente a partir de la tabla analítica y del diccionario de indicadores.
    - La validación cruzada entre SPSS y R debe realizarse utilizando una tolerancia numérica y no igualdad exacta.

    ## Error técnico que no quiero repetir

    - Asumir la existencia de variables de diseño muestral (estrato y conglomerado) sin verificarlas previamente en el esquema de datos.

    ## Riesgo mitigado

    - El notebook detiene la ejecución cuando faltan variables críticas, evitando generar resultados estadísticos incorrectos.
    """).strip(),
}

for sprint, text in retrospectives.items():
    path = LOG_DIR / f"stage3_sprint{sprint}_retro.md"
    path.write_text(text, encoding="utf-8")

print("Sprint retrospectives generated successfully.")

Sprint retrospectives generated successfully.
